
# Seminario de Estadística II - Tarea 2 Parte A

**Integrantes del equipo:**
* Azpeitia Medina Samuel
* Castro Pérez Juan Antonio
* Rodríguez Rodríguez Donovan Zuriel 

In [0]:
#Primero instalamos las bibliotecas necesarias
%pip install databricks-feature-engineering

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Reiniciamos el proceso de Python para que reconozca las instalaciones
dbutils.library.restartPython()

### 1. Realice un análisis de la calidad de los datos de bronze a silver, elimine duplicados, conversión de tipos, missing vs ceros, lo que crea pertinente.

**Solución:**
Para mejorar la calidad de nuestra tabla silver, lo primero que vamos a hacer es quitar cualquier fila que esté duplicada. Después, nos enfocaremos en las columnas numéricas como los bytes y los paquetes. En este tipo de datos, si hay un valor nulo, lo más correcto es ponerle un cero, ya que significa que no hubo tráfico detectado. Al final, nos aseguramos de que todos estos campos sean números enteros (long) para que no fallen los promedios más adelante.

In [0]:
import pyspark.sql.functions as F

# Cargamos la tabla silver con la estructura que ya tenemos
df_silver = spark.sql("SELECT * FROM dev.ciencias_data.silver_sessions")

# 1. Quitamos duplicados
df_calidad = df_silver.dropDuplicates()

# 2. Definimos las columnas numericas reales que aparecen en nuestra tabla
columnas_numericas = ["tot_packets", "src_packets", "dst_packets", "tot_bytes", "src_bytes", "dst_bytes", "tot_data_bytes"]

# 3. Rellenamos los nulos con 0 y convertimos a tipo long
df_calidad = df_calidad.fillna(0, subset=columnas_numericas)

for nombre_col in columnas_numericas:
    df_calidad = df_calidad.withColumn(nombre_col, F.col(nombre_col).cast("long"))

# Guardamos la tabla corregida sobreescribiendo la silver
df_calidad.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.ciencias_data.silver_sessions")

print("Calidad de datos terminada. Se limpiaron duplicados y se ajustaron los nulos en las columnas numericas.")
df_calidad.limit(20).display()

Calidad de datos terminada. Se limpiaron duplicados y se ajustaron los nulos en las columnas numericas.


timestamp,part_hour,src_ip,dst_ip,src_port,dst_port,src_mac,dst_mac,protocol,tot_packets,src_packets,dst_packets,tot_bytes,src_bytes,dst_bytes,tot_data_bytes,first_packet,last_packet,src_geo,dst_geo,http,packet_len_suma,packet_len_min,packet_len_max,packet_len_media
2020-03-23 18:24:18.003,2020-03-23-18,10.33.170.254,172.16.239.8,64015,8080,List(e8:1c:ba:f7:5f:72),List(e8:1c:ba:f0:32:5a),List(tcp),14,8,6,1252,480,772,0,2020-03-24T00:24:12.460Z,2020-03-24T00:24:12.486Z,null,null,null,1476,0,282,98.4
2020-03-23 23:07:08.009,2020-03-23-23,10.33.170.206,96.45.33.73,60058,8888,List(6c:0b:84:66:7c:e4),List(e8:1c:ba:f7:5f:72),List(udp),4,2,2,368,212,156,336,2020-03-24T05:06:37.784Z,2020-03-24T05:06:37.858Z,null,US,null,432,0,122,86.4
2020-03-23 18:24:20.002,2020-03-23-18,2620:0:cc6::169,2801:c4:15:200::ef,62651,53,List(2c:21:72:d0:78:f6),List(44:2b:03:53:d7:80),"List(udp, dns)",4,2,2,590,220,370,558,2020-03-24T00:23:49.079Z,2020-03-24T00:23:49.081Z,US,MX,null,654,0,201,130.8
2020-03-23 23:07:17.006,2020-03-23-23,10.10.8.22,10.20.1.250,0,0,List(00:08:e3:ff:fc:28),List(84:8a:8d:94:82:46),List(icmp),10,5,5,980,490,490,0,2020-03-24T05:07:02.368Z,2020-03-24T05:07:06.375Z,null,null,null,1140,0,114,103.63636363636364
2020-03-23 18:24:20.004,2020-03-23-18,10.10.16.197,10.33.255.108,50193,161,List(00:08:e3:ff:fc:28),List(00:23:eb:36:45:c1),"List(udp, snmp)",2,1,1,177,88,89,161,2020-03-24T00:23:49.161Z,2020-03-24T00:23:49.162Z,null,null,null,209,0,105,69.66666666666667
2020-03-23 23:07:19.003,2020-03-23-23,10.34.3.22,10.33.242.220,123,123,List(00:15:fa:c9:88:8d),List(40:ce:24:8a:ea:80),"List(udp, ntp)",2,2,0,180,180,0,164,2020-03-24T05:06:48.469Z,2020-03-24T05:06:48.469Z,null,null,null,212,0,106,70.66666666666667
2020-03-23 23:07:21.003,2020-03-23-23,200.78.173.253,187.189.183.202,47336,10010,List(4c:b1:6c:ed:1e:0d),List(94:10:3e:9c:86:bc),"List(http, tcp)",22,12,10,2892,1198,1694,786,2020-03-24T05:07:15.598Z,2020-03-24T05:07:15.790Z,MX,MX,List(List(187.189.183.202)),3244,0,617,141.04347826086956
2020-03-23 18:24:22.005,2020-03-23-18,10.33.224.203,52.185.211.133,63139,443,List(00:08:e3:ff:fc:28),List(00:09:0f:09:02:08),"List(tls, tcp)",44,22,22,13426,3996,9430,5447,2020-03-24T00:24:16.583Z,2020-03-24T00:24:16.739Z,null,US,List(List(settings-win.data.microsoft.com)),14130,0,1470,314.0
2020-03-23 23:07:28.007,2020-03-23-23,10.40.130.58,190.2.151.204,54226,80,"List(00:08:e3:ff:fc:28, 00:af:1f:60:9a:cd)","List(00:08:e3:ff:fc:28, 00:09:0f:09:02:08)","List(http, tcp)",27,15,12,3309,1674,1635,571,2020-03-24T05:07:21.776Z,2020-03-24T05:07:22.132Z,null,NL,List(List(b49.cleverjumper.com)),3741,0,379,133.60714285714286
2020-03-23 23:07:28.007,2020-03-23-23,2.5.5.2,10.128.0.1,26221,53,List(00:09:0f:09:00:09),List(00:09:0f:09:02:09),"List(udp, dns)",4,2,2,474,162,312,442,2020-03-24T05:06:57.342Z,2020-03-24T05:06:57.342Z,FR,null,null,538,0,172,107.6


### 2. Construya un FeatureStore para los datos de sesiones usando FeatureEngineeringClient() en databricks.

* **Construya las variables que crea pertinentes como duración de la sesión, bytes trasmitidos por unidad de tiempo, el tamaño promedio de los paquetes, ratio de bytes recibidos entre enviados etc.**

**Solución:**
En este paso vamos a crear las "features" que nos pide el ejercicio. Vamos a calcular cuánto duró la sesión restando el tiempo del primer y último paquete. También sacaremos el tamaño promedio de los paquetes y la proporción de bytes que se recibieron contra los que se enviaron. Como el Feature Store nos pide una llave primaria, vamos a generar un ID único (UUID) para cada registro de la tabla.

In [0]:
import uuid
from pyspark.sql.types import StringType

# Volvemos a leer la tabla silver ya limpia
df_entrenamiento = spark.sql("SELECT * FROM dev.ciencias_data.silver_sessions")

# Calculamos la duracion de la sesion en segundos
# Usamos to_timestamp para que Spark entienda el formato ISO que traen las columnas
df_entrenamiento = df_entrenamiento.withColumn("duracion_seg", F.unix_timestamp(F.to_timestamp("last_packet")) - F.unix_timestamp(F.to_timestamp("first_packet")))

# Ratio de bytes (destino / origen)
df_entrenamiento = df_entrenamiento.withColumn("ratio_recibido_enviado", F.when(F.col("src_bytes") > 0, F.col("dst_bytes") / F.col("src_bytes")).otherwise(0.0))

# Tamaño promedio de los paquetes
df_entrenamiento = df_entrenamiento.withColumn("promedio_tamanio_paquete", F.when(F.col("tot_packets") > 0, F.col("tot_bytes") / F.col("tot_packets")).otherwise(0.0))

# Creamos el UUID para usarlo como Primary Key
def crear_identificador():
    return str(uuid.uuid4())

udf_id = F.udf(crear_identificador, StringType())
df_features_final = df_entrenamiento.withColumn("id", udf_id())

print("Variables para el Feature Store calculadas con exito.")
df_features_final.select("id", "duracion_seg", "ratio_recibido_enviado", "promedio_tamanio_paquete").limit(20).display()

Variables para el Feature Store calculadas con exito.


id,duracion_seg,ratio_recibido_enviado,promedio_tamanio_paquete
97664b55-6958-477b-9900-b3a95a7bc715,0,1.6083333333333334,89.42857142857143
e0eb9347-790f-41b0-8e7e-648e2334465d,0,0.7358490566037735,92.0
af35336f-bcd9-4b82-b9cf-e8dc6a18b26d,0,1.6818181818181819,147.5
00feccaa-881e-4091-a5f1-f2eaac93b72b,4,1.0,98.0
dee9cca2-4d99-4633-a20e-adf283ff2f9c,0,1.0113636363636365,88.5
2ecf711e-7629-4d20-9e5b-873e1ca05ccc,0,0.0,90.0
ec1c73a9-a72c-4bc0-8ed2-5e04d18dee93,0,1.4140233722871451,131.45454545454547
57a3d5eb-5dd7-4fbd-8fb9-1be748565c64,0,2.35985985985986,305.1363636363636
30477a7a-8c4e-463d-8536-f3a172d9683e,1,0.9767025089605734,122.55555555555556
429b3ff4-8f68-4d7f-b0a9-1383ba3ee6a5,0,1.9259259259259258,118.5


* **Registre el FeatureStore en Unity Catalog como se vio en clase.**

**Solución:**
Para terminar esta parte, vamos a registrar nuestro conjunto de datos en el Feature Store de Databricks. Primero creamos el esquema `feature_store` para que todo esté organizado. Después, usamos el `FeatureEngineeringClient` para crear la tabla oficial dentro de Unity Catalog, asignando la columna `id` que creamos antes como nuestra llave primaria.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

# Creamos la base de datos para el almacén de características
spark.sql("CREATE SCHEMA IF NOT EXISTS dev.feature_store")

# Iniciamos el cliente de ingenieria de caracteristicas
cliente_fe = FeatureEngineeringClient()

# Registramos la tabla de caracteristicas en Unity Catalog
cliente_fe.create_table(
    name="dev.feature_store.sessions_arkime",
    primary_keys=["id"],
    df=df_features_final,
    schema=df_features_final.schema,
    description="Tabla con las variables de duracion, ratios y promedios para el modelo de ML"
)

print("El Feature Store ha sido registrado exitosamente en dev.feature_store.sessions_arkime")

El Feature Store ha sido registrado exitosamente en dev.feature_store.sessions_arkime


### 3. Cree un Pipeline de datos donde implemente las diferentes técnicas de Feature Engineering vistos en la clase.
* **Construya el “set de entrenamiento” usando FeatureLookup y FeatureEngineeringClient.**

**Solución:**
Para empezar a modelar, no vamos a usar la tabla directamente. Siguiendo lo que vimos en clase, vamos a crear un set de entrenamiento usando `FeatureLookup`. Esto nos permite "ir a buscar" las variables que registramos en el Feature Store (como la duración y los ratios) usando únicamente nuestra columna `id` como llave. Esto es súper útil porque así mantenemos el linaje de los datos.

In [0]:
from databricks.feature_engineering import FeatureLookup


df_base_ids = spark.table("dev.feature_store.sessions_arkime").select("id")

# 2. El resto se queda igual
model_lookups = [
    FeatureLookup(
        table_name="dev.feature_store.sessions_arkime",
        lookup_key="id"
    )
]

# 3. Creamos el set usando 'cliente_fe'
set_entrenamiento = cliente_fe.create_training_set(
    df=df_base_ids,
    feature_lookups=model_lookups,
    label=None,
    exclude_columns=["id"] # Aquí solo excluimos el id
)

df_entrenar = set_entrenamiento.load_df()

df_entrenar.limit(20).display()

timestamp,part_hour,src_ip,dst_ip,src_port,dst_port,src_mac,dst_mac,protocol,tot_packets,src_packets,dst_packets,tot_bytes,src_bytes,dst_bytes,tot_data_bytes,first_packet,last_packet,src_geo,dst_geo,http,packet_len_suma,packet_len_min,packet_len_max,packet_len_media,duracion_seg,ratio_recibido_enviado,promedio_tamanio_paquete
2020-03-23 18:24:38.003,2020-03-23-18,2620:0:cc6::167,2801:c4:15:200::e1,59323,53,List(2c:21:72:d0:78:f6),List(44:2b:03:53:d7:80),"List(udp, dns)",2,2,0,214,214,0,198,2020-03-24T00:24:07.663Z,2020-03-24T00:24:07.663Z,US,MX,null,246,0,123,82.0,0,0.0,107.0
2020-03-23 23:07:28.007,2020-03-23-23,10.40.130.58,190.2.151.204,54226,80,"List(00:08:e3:ff:fc:28, 00:af:1f:60:9a:cd)","List(00:08:e3:ff:fc:28, 00:09:0f:09:02:08)","List(http, tcp)",27,15,12,3309,1674,1635,571,2020-03-24T05:07:21.776Z,2020-03-24T05:07:22.132Z,null,NL,List(List(b49.cleverjumper.com)),3741,0,379,133.60714285714286,1,0.9767025089605734,122.55555555555556
2020-03-23 23:07:19.003,2020-03-23-23,10.34.3.22,10.33.242.220,123,123,List(00:15:fa:c9:88:8d),List(40:ce:24:8a:ea:80),"List(udp, ntp)",2,2,0,180,180,0,164,2020-03-24T05:06:48.469Z,2020-03-24T05:06:48.469Z,null,null,null,212,0,106,70.66666666666667,0,0.0,90.0
2020-03-23 22:10:59.008,2020-03-23-22,10.33.225.70,172.217.9.3,61147,443,List(00:08:e3:ff:fc:28),List(00:09:0f:09:02:08),List(tcp),10,6,4,612,360,252,0,2020-03-24T04:10:24.596Z,2020-03-24T04:10:53.147Z,null,US,null,772,0,82,70.18181818181819,29,0.7,61.2
2020-03-23 22:11:02.002,2020-03-23-22,10.33.120.175,168.61.144.182,49518,443,"List(00:08:e3:ff:fc:28, 00:09:0f:09:02:12)","List(00:08:e3:ff:fc:28, 00:09:0f:09:02:08)","List(tls, tcp)",66,27,39,17106,3444,13662,4436,2020-03-24T04:10:56.154Z,2020-03-24T04:10:56.426Z,null,US,List(List(j5yrru.manage.trendmicro.com)),18162,0,1464,271.07462686567163,0,3.966898954703833,259.1818181818182
2020-03-23 23:07:21.003,2020-03-23-23,200.78.173.253,187.189.183.202,47336,10010,List(4c:b1:6c:ed:1e:0d),List(94:10:3e:9c:86:bc),"List(http, tcp)",22,12,10,2892,1198,1694,786,2020-03-24T05:07:15.598Z,2020-03-24T05:07:15.790Z,MX,MX,List(List(187.189.183.202)),3244,0,617,141.04347826086956,0,1.4140233722871451,131.45454545454547
2020-03-23 23:07:35.006,2020-03-23-23,192.151.112.162,187.188.92.141,55179,10009,List(00:08:e3:ff:fc:28),List(00:09:0f:09:02:08),"List(http, tcp)",22,10,12,2764,930,1834,726,2020-03-24T05:07:29.618Z,2020-03-24T05:07:29.736Z,US,MX,List(List(187.188.92.141)),3116,0,631,135.47826086956522,0,1.972043010752688,125.63636363636364
2020-03-23 18:24:45.006,2020-03-23-18,10.33.170.222,172.16.239.8,49585,8080,"List(6c:0b:84:66:7e:94, e8:1c:ba:f7:5f:72)","List(e8:1c:ba:f0:32:5a, e8:1c:ba:f7:5f:72)","List(http, tcp)",32,22,10,3738,1940,1798,759,2020-03-24T00:24:39.404Z,2020-03-24T00:24:39.412Z,null,null,"List(List(172.16.239.8:8080, 172.16.239.8))",4250,0,669,128.78787878787878,0,0.9268041237113402,116.8125
2020-03-23 23:07:08.009,2020-03-23-23,10.33.170.206,96.45.33.73,60058,8888,List(6c:0b:84:66:7c:e4),List(e8:1c:ba:f7:5f:72),List(udp),4,2,2,368,212,156,336,2020-03-24T05:06:37.784Z,2020-03-24T05:06:37.858Z,null,US,null,432,0,122,86.4,0,0.7358490566037735,92.0
2020-03-23 23:07:28.007,2020-03-23-23,2.5.5.2,10.128.0.1,26221,53,List(00:09:0f:09:00:09),List(00:09:0f:09:02:09),"List(udp, dns)",4,2,2,474,162,312,442,2020-03-24T05:06:57.342Z,2020-03-24T05:06:57.342Z,FR,null,null,538,0,172,107.6,0,1.9259259259259258,118.5


* **Realize un análisis de conglomerados, usando las diferentes técnicas como K-Means y DBSCAN.**

**Solución:**
Ahora vamos a agrupar nuestras sesiones de red en diferentes categorías usando el algoritmo **K-Means**. Como las variables que traemos del Feature Store tienen escalas muy distintas (por ejemplo, la duración está en segundos y los bytes son números gigantes), vamos a crear un `Pipeline` que primero ensamble todas las columnas en un vector y luego las normalice usando `StandardScaler`. Esto es indispensable para que el modelo calcule bien las distancias y los grupos (clusters) sean precisos.

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

# 1. Definimos las variables numericas que vamos a usar para agrupar
# Usamos las que vienen de nuestra tabla de features
columnas_modelo = ["duracion_seg", "promedio_tamanio_paquete", "tot_packets", "tot_bytes", "tot_data_bytes"]

# 2. Configuramos el ensamblador para crear el vector de entrada
ensamblador = VectorAssembler(inputCols=columnas_modelo, outputCol="features_sin_escalar", handleInvalid="skip")

# 3. Escalamos los datos para que tengan media 0 y varianza 1
escalador = StandardScaler(inputCol="features_sin_escalar", outputCol="features", withStd=True, withMean=True)

# 4. Definimos el modelo K-Means. Vamos a probar con 3 grupos (clusters)
modelo_km = KMeans(k=3, seed=42, featuresCol="features", predictionCol="grupo_cluster")

# 5. Creamos y entrenamos el Pipeline completo
pipeline_agrupamiento = Pipeline(stages=[ensamblador, escalador, modelo_km])
modelo_entrenado = pipeline_agrupamiento.fit(df_entrenar)

# Aplicamos el modelo al set de entrenamiento
df_con_grupos = modelo_entrenado.transform(df_entrenar)

print("Sesiones agrupadas por comportamiento:")
df_con_grupos.select("src_ip", "dst_ip", "duracion_seg", "tot_bytes", "grupo_cluster").limit(20).display()

Sesiones agrupadas por comportamiento:


src_ip,dst_ip,duracion_seg,tot_bytes,grupo_cluster
10.33.170.254,172.16.239.8,0,1252,0
10.33.170.206,96.45.33.73,0,368,0
2620:0:cc6::169,2801:c4:15:200::ef,0,590,0
10.10.8.22,10.20.1.250,4,980,0
10.10.16.197,10.33.255.108,0,177,0
10.34.3.22,10.33.242.220,0,180,0
200.78.173.253,187.189.183.202,0,2892,0
10.33.224.203,52.185.211.133,0,13426,0
10.40.130.58,190.2.151.204,1,3309,0
2.5.5.2,10.128.0.1,0,474,0


* **Implemente un modelo de detecci ́on de anomalías usando Isolation Forest.**

**Solución:**

In [0]:
# Código

* **Comparta sus conclusiones de manera detallada, incluyendo como determino los perfiles de comportamiento y grupos.**

**Solución:**

In [0]:
# Código

* **Valide la calidad de su clusterización, ¿Qué observa?**

**Solución:**

### 4. Para el ejemplo visto en clase sobre clustering jerárquico, realice el mismo ejemplo utilizando el enlace completo.

**Solución:**

Para el método de enlace completo, la distancia entre dos clústeres se toma como la distancia máxima entre cualquier par de elementos de ambos grupos:

d(A,B) = max { d(xi, xj) } para xi en A, xj en B.

Empezamos con la matriz de distancias original, donde cada delegación es un clúster individual:

| | Milpa Alta | Tláhuac | Iztapalapa | Tlalpan | Xochimilco | Coyoacán |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Milpa Alta | 0 | 33.2 | 31.3 | 25.7 | 11.4 | 39.6 |
| Tláhuac | 33.2 | 0 | 8.6 | 18.5 | 15.8 | 15.4 |
| Iztapalapa | 31.3 | 8.6 | 0 | 18.7 | 16.0 | 15.3 |
| Tlalpan | 25.7 | 18.5 | 18.7 | 0 | 9.3 | 11.1 |
| Xochimilco | 11.4 | 15.8 | 16.0 | 9.3 | 0 | 15.3 |
| Coyoacán | 39.6 | 15.4 | 15.3 | 11.1 | 15.3 | 0 |

La distancia mínima en la matriz es 8.6, que corresponde a Tláhuac e Iztapalapa. Los agrupamos formando el clúster {Tláhuac, Iztapalapa}.

Recalculamos las distancias de este nuevo clúster hacia los demás usando siempre el valor máximo:
* d({Tláhuac, Iztap.}, MA) = max(33.2, 31.3) = 33.2
* d({Tláhuac, Iztap.}, Tlalpan) = max(18.5, 18.7) = 18.7
* d({Tláhuac, Iztap.}, Xochimilco) = max(15.8, 16.0) = 16.0
* d({Tláhuac, Iztap.}, Coyoacán) = max(15.4, 15.3) = 15.4

Matriz actualizada:

| | {Tláhuac, Iztap.} | Milpa Alta | Tlalpan | Xochimilco | Coyoacán |
|:---|:---:|:---:|:---:|:---:|:---:|
| {Tláhuac, Iztap.}| 0 | 33.2 | 18.7 | 16.0 | 15.4 |
| Milpa Alta | 33.2 | 0 | 25.7 | 11.4 | 39.6 |
| Tlalpan | 18.7 | 25.7 | 0 | 9.3 | 11.1 |
| Xochimilco | 16.0 | 11.4 | 9.3 | 0 | 15.3 |
| Coyoacán | 15.4 | 39.6 | 11.1 | 15.3 | 0 |

Ahora la nueva distancia mínima en la matriz es 9.3, entre Tlalpan y Xochimilco. Los agrupamos en el clúster {Tlalpan, Xochimilco}.

Recalculamos las distancias:
* d({Tlalpan, Xoch.}, {Tláhuac, Iztap.}) = max(18.7, 16.0) = 18.7
* d({Tlalpan, Xoch.}, MA) = max(25.7, 11.4) = 25.7
* d({Tlalpan, Xoch.}, Coyoacán) = max(11.1, 15.3) = 15.3

Matriz actualizada:

| | {Tláhuac, Iztap.} | {Tlalpan, Xoch.} | Milpa Alta | Coyoacán |
|:---|:---:|:---:|:---:|:---:|
| {Tláhuac, Iztap.}| 0 | 18.7 | 33.2 | 15.4 |
| {Tlalpan, Xoch.} | 18.7 | 0 | 25.7 | 15.3 |
| Milpa Alta | 33.2 | 25.7 | 0 | 39.6 |
| Coyoacán | 15.4 | 15.3 | 39.6 | 0 |

Buscamos de nuevo y la distancia mínima es 15.3, entre el clúster {Tlalpan, Xochimilco} y Coyoacán. Los unimos en {Tlalpan, Xochimilco, Coyoacán}.

Recalculamos las distancias:
* d({Tlalpan, Xoch., Coy.}, {Tláhuac, Iztap.}) = max(18.7, 15.4) = 18.7
* d({Tlalpan, Xoch., Coy.}, MA) = max(25.7, 39.6) = 39.6

Matriz actualizada:

| | {Tláhuac, Iztap.} | {Tlalpan, Xoch., Coy.} | Milpa Alta |
|:---|:---:|:---:|:---:|
| {Tláhuac, Iztap.}| 0 | 18.7 | 33.2 |
| {Tlalpan, Xoch., Coy.} | 18.7 | 0 | 39.6 |
| Milpa Alta | 33.2 | 39.6 | 0 |

En este punto la distancia mínima es 18.7, que corresponde a los clústeres {Tláhuac, Iztapalapa} y {Tlalpan, Xochimilco, Coyoacán}. Los juntamos en un solo grupo.

Calculamos la distancia restante con Milpa Alta:
* d({Tláhuac, Iztap., Tlalpan, Xoch., Coy.}, MA) = max(33.2, 39.6) = 39.6

Matriz actualizada:

| | {Tláhuac, Iztap., Tlalpan, Xoch., Coy.} | Milpa Alta |
|:---|:---:|:---:|
| {Tláhuac, Iztap., Tlalpan, Xoch., Coy.}| 0 | 39.6 |
| Milpa Alta | 39.6 | 0 |

Conclusión:

Si el gobierno de la CDMX requiere poner exactamente dos oficinas administrativas, nos detenemos antes de la última agrupación, lo que nos deja con los siguientes dos grupos:

Oficina 1: Tláhuac, Iztapalapa, Tlalpan, Xochimilco y Coyoacán.

Oficina 2: Milpa Alta.
